# Response Time Analysis — Thread-Based (April 2026)
**Input:** `2026_04_17_messages.xls` + `2026_04_17_reviews.xls`  
**Output:**
- `booking_response.parquet` — per-booking aggregated reply stats
- `daily_response.parquet` — daily median reply time per hotel (MrBeast chart)
- `hotel_reviews.parquet` — review stats per hotel
- `hotel_summary_v2.parquet` — per-hotel summary for Tab 1

---
### Как считается время ответа
**Проблема наивного подхода:** брать первое сообщение гостя за всю историю брони → ответ может найтись через 3 месяца в другом разговоре → ложные 89 дней.

**Правильный подход (thread-based):**
1. Разбить переписку на треды по паузам > 4 часов (как в `hotel_threads.ipynb`)
2. Для каждого треда: время от первого гостевого сообщения до первого ответа admin
3. Ответ позже 24 часов → `NaN` (не ответили на этот конкретный тред)
4. По брони: медиана по всем тредам

In [8]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data')
MSG_PATH = DATA_DIR / '2026_04_17_messages.xls'
REV_PATH = DATA_DIR / '2026_04_17_reviews.xls'

HOTEL_NAMES    = {1: 'БС74', 2: 'МК16', 3: 'Дубай', 4: 'М73', 5: 'О-44'}
GAP_HOURS      = 4    # пауза > 4ч = новый тред
REPLY_WINDOW_H = 48   # ответ позже 24ч не считается ответом на тред

print('Ready.')

Ready.


In [9]:
# ── Cell 1: Load & clean messages ─────────────────────────────────────────────
msg = pd.read_excel(MSG_PATH, engine='xlrd')
msg['date_add']   = pd.to_datetime(msg['date_add'])
msg['is_admin']   = msg['from_admin'].fillna(0).astype(int)
msg['message']    = msg['message'].astype(str).str.strip()
msg = msg.rename(columns={'booking_id': 'ID_booking'})
msg['hotel_name'] = msg['hotel_id'].map(HOTEL_NAMES)

# Drop blank / test / HTML-only rows
SKIP = r'^(nan|тест|test|бегу|file|\s*)$'
msg = msg[~msg['message'].str.lower().str.match(SKIP)].copy()
msg = msg[msg['message'].str.len() >= 1].copy()
msg = msg.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)

print(f"Rows:     {len(msg):,}")
print(f"Hotels:   {sorted(msg['hotel_id'].unique())}")
print(f"Period:   {msg['date_add'].min().date()} → {msg['date_add'].max().date()}")
print(f"Bookings: {msg['ID_booking'].nunique():,}")
print()
print(msg.groupby('hotel_name').agg(
    messages  = ('message', 'size'),
    bookings  = ('ID_booking', 'nunique'),
    guest_msg = ('is_admin', lambda x: (x == 0).sum()),
    admin_msg = ('is_admin', lambda x: (x == 1).sum()),
))

Rows:     64,294
Hotels:   [1, 2, 3, 4]
Period:   2022-03-31 → 2026-04-17
Bookings: 18,577

            messages  bookings  guest_msg  admin_msg
hotel_name                                          
БС74           31240      8025      15484      15756
Дубай          15576      5498       4805      10771
М73              871       209         98        773
МК16           16607      4845       5257      11350


In [10]:
# ── Cell 2: Thread detection ──────────────────────────────────────────────────
# Новый тред = пауза > GAP_HOURS между любыми двумя сообщениями в брони.
# Запускаем на ВСЕХ сообщениях (включая admin), чтобы паузы считались правильно.

msg['prev_time']  = msg.groupby('ID_booking')['date_add'].shift(1)
msg['gap_hrs']    = (msg['date_add'] - msg['prev_time']).dt.total_seconds() / 3600
msg['new_thread'] = msg['prev_time'].isna() | (msg['gap_hrs'] > GAP_HOURS)
msg['thread_id']  = msg.groupby('ID_booking')['new_thread'].cumsum()

n_threads   = msg.groupby(['ID_booking', 'thread_id']).ngroups
n_bookings  = msg['ID_booking'].nunique()
thread_sizes = msg.groupby(['ID_booking', 'thread_id']).size()

print(f"Тредов:             {n_threads:,}")
print(f"Броней:             {n_bookings:,}")
print(f"Среднее на бронь:   {n_threads / n_bookings:.1f}")
print(f"Однострочные треды: {(thread_sizes == 1).sum():,}")
print(f"Многострочные:      {(thread_sizes > 1).sum():,}")

Тредов:             30,923
Броней:             18,577
Среднее на бронь:   1.7
Однострочные треды: 19,308
Многострочные:      11,615


In [11]:
# ── Cell 3: Thread reply time ─────────────────────────────────────────────────
# Для каждого треда:
#   thread_start = время первого ГОСТЕВОГО сообщения в треде
#   admin_reply  = время первого admin-сообщения в том же треде
#   reply_min    = разница в минутах
#
# Важно: admin_reply берём из ТОГО ЖЕ треда (group by ID_booking + thread_id),
# поэтому ответ из соседнего треда не попадёт сюда.

guest_msg = msg[msg['is_admin'] == 0]
admin_msg = msg[msg['is_admin'] == 1]

# First guest message per thread
thread_start = (
    guest_msg
    .groupby(['ID_booking', 'thread_id'])['date_add']
    .min()
    .rename('thread_start')
    .reset_index()
)

# First admin message per thread
thread_admin = (
    admin_msg
    .groupby(['ID_booking', 'thread_id'])['date_add']
    .min()
    .rename('admin_reply')
    .reset_index()
)

threads = thread_start.merge(thread_admin, on=['ID_booking', 'thread_id'], how='left')

# Add hotel
hotel_per_booking = msg.groupby('ID_booking')['hotel_id'].first()
threads = threads.join(hotel_per_booking, on='ID_booking')
threads['hotel_name'] = threads['hotel_id'].map(HOTEL_NAMES)

# Reply time in minutes
threads['reply_min'] = (
    (threads['admin_reply'] - threads['thread_start']).dt.total_seconds() / 60
)

# Admin replied before guest's first message in thread → artifact, drop
threads.loc[threads['reply_min'] <= 0, 'reply_min'] = np.nan

# Reply after 24h → too late to count as response to this thread
threads.loc[threads['reply_min'] > REPLY_WINDOW_H * 60, 'reply_min'] = np.nan

# Keep only threads that had at least one guest message
threads = threads[threads['thread_start'].notna()].copy()

n_total   = len(threads)
n_replied = threads['reply_min'].notna().sum()
print(f"Тредов с гостевыми сообщениями: {n_total:,}")
print(f"Получили ответ (< 24ч):         {n_replied:,} ({n_replied/n_total*100:.1f}%)")
print()
print(threads.groupby('hotel_name')['reply_min'].agg(
    n_threads = 'size',
    replied   = 'count',
    median    = 'median',
    mean      = 'mean',
    p90       = lambda x: x.quantile(0.9),
).round(1))

Тредов с гостевыми сообщениями: 12,407
Получили ответ (< 24ч):         6,269 (50.5%)

            n_threads  replied  median  mean    p90
hotel_name                                         
БС74             7017     4314    13.7  36.5  113.6
Дубай            2579      752    29.8  48.9  126.4
М73                50       18     3.1  10.7   31.7
МК16             2761     1185     3.3  15.2   37.2


In [12]:
# ── Cell 4: Sanity check — verify on real examples ───────────────────────────
# Смотрим реальные треды и проверяем что время посчитано правильно

sample = threads[threads['reply_min'].notna()].sample(5, random_state=42)

for _, row in sample.iterrows():
    sub = msg[
        (msg['ID_booking'] == row['ID_booking']) &
        (msg['thread_id']  == row['thread_id'])
    ][['date_add', 'is_admin', 'message']].sort_values('date_add')

    print(f"[{row['hotel_name']} / бронь {row['ID_booking']} / тред {row['thread_id']}]  "
          f"reply = {row['reply_min']:.1f} мин")
    for _, m in sub.iterrows():
        who = 'ADMIN' if m['is_admin'] else 'GUEST'
        print(f"  {m['date_add'].strftime('%Y-%m-%d %H:%M')}  {who:<5}  {str(m['message'])[:80]}")
    print()

[БС74 / бронь 1203203 / тред 2]  reply = 1.9 мин
  2024-04-06 20:01  GUEST  Добрый вечер, подскажите код ваучера номер 1431.
  2024-04-06 20:03  ADMIN  Добрый вечер. 
94177-56361
  2024-04-06 20:04  GUEST  Спасибо

[БС74 / бронь 1173555 / тред 14]  reply = 17.3 мин
  2024-03-01 23:15  GUEST  Здравствуйте, вчера у нас была уборка, сегодня заметила, что на одном большом по
  2024-03-01 23:33  ADMIN  Доброй ночи! Ожидайте, подойдут к вам

[МК16 / бронь 21363417 / тред 2]  reply = 33.4 мин
  2025-05-29 20:55  GUEST  Добрый день
  2025-05-29 20:55  GUEST  А завтрак со скольки до скольки?
  2025-05-29 21:28  ADMIN  Добрый с 7.30 до 11.00
  2025-05-29 22:16  GUEST  А музыка с басами наверху до скольки?!!!???
  2025-05-29 22:39  ADMIN  сейчас выключит бар, извините
  2025-05-29 22:40  GUEST  Ок спасиьо
  2025-05-29 22:40  GUEST  Ок спасибо

[МК16 / бронь 2934667 / тред 2]  reply = 4.5 мин
  2024-03-09 10:57  GUEST  Помогите, пожалуйста, номер открыть 310, я карту внутри забыла
  2024-03-09 10:

In [13]:
# ── Cell 5: Per-booking aggregation ──────────────────────────────────────────
# По каждой брони: медиана времени ответа по всем тредам,
# + первое гостевое сообщение (для временной оси в графике)

booking_first_guest = (
    guest_msg
    .groupby('ID_booking')['date_add']
    .min()
    .rename('first_guest_time')
)

br = (
    threads
    .groupby(['ID_booking', 'hotel_id', 'hotel_name'])
    .agg(
        reply_time_min = ('reply_min', 'median'),  # медиана по тредам брони
        n_threads      = ('thread_id', 'nunique'),
        n_replied      = ('reply_min', 'count'),
    )
    .reset_index()
    .join(booking_first_guest, on='ID_booking')
)

br['year'] = br['first_guest_time'].dt.year
br['date'] = br['first_guest_time'].dt.date

# Speed buckets (same as old dashboard)
def bucket(m):
    if pd.isna(m): return np.nan
    if m <= 5:     return '<=5m'
    if m <= 15:    return '5-15m'
    if m <= 60:    return '15-60m'
    return '>60m'
br['reply_bucket'] = br['reply_time_min'].apply(bucket)

print(f"Броней: {len(br):,}")
print()
print(br.groupby('hotel_name')['reply_time_min'].agg(
    bookings    = 'size',
    no_reply    = lambda x: x.isna().sum(),
    resp_rate   = lambda x: f"{x.notna().mean()*100:.1f}%",
    median      = 'median',
    p90         = lambda x: x.quantile(0.9),
).round(1))

Броней: 6,543

            bookings  no_reply resp_rate  median    p90
hotel_name                                             
БС74            3475      1227     64.7%    13.9  106.2
Дубай           1246       801     35.7%    30.0  112.0
М73               39        23     41.0%     3.4   36.3
МК16            1783       915     48.7%     3.3   35.5


In [14]:
# ── Cell 6: Daily aggregation (for MrBeast-style chart) ──────────────────────
# Каждая строка = (отель, день) → медиана времени ответа в этот день
# rolling_30d = 30-дневное скользящее среднее = красная линия тренда

daily = (
    br.dropna(subset=['reply_time_min'])
      .groupby(['hotel_id', 'hotel_name', 'date'])
      .agg(
          median_reply = ('reply_time_min', 'median'),
          mean_reply   = ('reply_time_min', 'mean'),
          n_bookings   = ('ID_booking', 'nunique'),
      )
      .reset_index()
)
daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values(['hotel_id', 'date'])

daily['rolling_30d'] = (
    daily.groupby('hotel_id')['median_reply']
         .transform(lambda s: s.rolling(30, min_periods=5).mean())
)

print(f"Daily rows: {len(daily):,}")
print(daily.groupby('hotel_name').agg(
    days           = ('date', 'nunique'),
    date_from      = ('date', 'min'),
    date_to        = ('date', 'max'),
    median_overall = ('median_reply', 'median'),
).round(1))

Daily rows: 1,777
            days  date_from    date_to  median_overall
hotel_name                                            
БС74         875 2022-11-01 2026-04-15            18.8
Дубай        329 2023-03-28 2026-04-15            33.4
М73           15 2023-04-23 2023-08-12             3.6
МК16         558 2023-03-28 2026-04-11             3.6


In [15]:
# ── Cell 7: Reviews ───────────────────────────────────────────────────────────
rev = pd.read_excel(REV_PATH, engine='xlrd')
rev['date_add']   = pd.to_datetime(rev['date_add'])
rev['hotel_name'] = rev['hotel_id'].map(HOTEL_NAMES)
rev['has_text']   = rev['comment'].notna() & (rev['comment'].astype(str).str.strip().str.len() > 0)

hotel_reviews = (
    rev.groupby(['hotel_id', 'hotel_name'])
       .agg(
           n_reviews      = ('kaizen_rating', 'size'),
           avg_rating     = ('kaizen_rating', 'mean'),
           n_with_text    = ('has_text', 'sum'),
           low_rating_pct = ('kaizen_rating', lambda s: (s <= 3).mean() * 100),
       )
       .round(2)
       .reset_index()
)
print(hotel_reviews)

   hotel_id hotel_name  n_reviews  avg_rating  n_with_text  low_rating_pct
0         1       БС74        351        4.52           83           11.68
1         2       МК16        132        4.71           42            6.82
2         3      Дубай         68        4.46           32           13.24
3         4        М73        151        4.62           28            9.93


In [16]:
# ── Cell 8: Hotel summary v2 ──────────────────────────────────────────────────
reply_stats = (
    br.groupby(['hotel_id', 'hotel_name'])['reply_time_min']
      .agg(
          n_bookings   = 'size',
          median_reply = 'median',
          mean_reply   = 'mean',
          p90_reply    = lambda x: x.quantile(0.9),
          no_reply     = lambda x: x.isna().sum(),
      ).round(1).reset_index()
)
reply_stats['response_rate'] = (
    (reply_stats['n_bookings'] - reply_stats['no_reply'])
    / reply_stats['n_bookings'] * 100
).round(1)

hotel_summary_v2 = reply_stats.merge(hotel_reviews, on=['hotel_id', 'hotel_name'], how='left')
print(hotel_summary_v2.to_string())

   hotel_id hotel_name  n_bookings  median_reply  mean_reply  p90_reply  no_reply  response_rate  n_reviews  avg_rating  n_with_text  low_rating_pct
0         1       БС74        3475          13.9        34.8      106.2      1227           64.7        351        4.52           83           11.68
1         2       МК16        1783           3.3        13.6       35.5       915           48.7        132        4.71           42            6.82
2         3      Дубай        1246          30.0        45.4      112.0       801           35.7         68        4.46           32           13.24
3         4        М73          39           3.4        11.1       36.3        23           41.0        151        4.62           28            9.93


In [17]:
# ── Cell 9: Save ──────────────────────────────────────────────────────────────
outputs = {
    'booking_response.parquet': br[[
        'ID_booking', 'hotel_id', 'hotel_name',
        'first_guest_time', 'reply_time_min', 'reply_bucket',
        'n_threads', 'n_replied', 'year', 'date'
    ]],
    'daily_response.parquet':   daily,
    'hotel_reviews.parquet':    hotel_reviews,
    'hotel_summary_v2.parquet': hotel_summary_v2,
}

for fname, df in outputs.items():
    path = DATA_DIR / fname
    df.to_parquet(path, index=False)
    print(f'✓  {fname:<35} {len(df):>6} rows  {path.stat().st_size/1024:.0f} KB')

✓  booking_response.parquet              6543 rows  144 KB
✓  daily_response.parquet                1777 rows  47 KB
✓  hotel_reviews.parquet                    4 rows  4 KB
✓  hotel_summary_v2.parquet                 4 rows  8 KB
